In [8]:
import pandas as pd
import numpy as np

# Indlæs filen
df_input = pd.read_csv('Data69/input_landbrugsdata.csv')
df_timer = pd.read_csv('Data69/Lontimer_landbrugsdata.csv')
#filter for løbende priser og foregående års priser
df_lonsum = df_input.loc[(df_input['TILGANG1'] == 'Aflønning af ansatte') & 
                         (df_input['PRISENHED'] == 'Løbende priser')].copy()
df_timerxselv = df_timer.loc[(df_timer['SOCIO'] == 'Præsterede timer for lønmodtagere (1000 timer)')].copy()
df_timer_ialt = df_timer.loc[(df_timer['SOCIO'] != 'Præsterede timer for lønmodtagere (1000 timer)')].copy()
#fjern prishenhed
df_lonsum.drop(columns=['PRISENHED'], inplace=True)
df_lonsum.drop(columns=['TILGANG1'], inplace=True)
df_timerxselv.drop(columns=['SOCIO'], inplace=True)
df_timer_ialt.drop(columns=['SOCIO'], inplace=True)

mapping = {
    '01000 Landbrug og gartneri': '01000',
    '01000 Landbrug og gartneri- (Anvendelse)': '01000',
    '10120 Føde-, drikke- og tobaksvareindustri': '10120',
    '10120 Føde-, drikke- og tobaksvareindustri- (Anvendelse)': '10120',
    'REST_TILGANG Øvrige brancher': 'REST',
    'REST_ANVENDELSE Øvrige brancher': 'REST'
}

# Erstat navnene i de to kolonner
df_timerxselv['BRANCHE'] = df_timerxselv['BRANCHE'].replace(mapping)
df_timer_ialt['BRANCHE'] = df_timer_ialt['BRANCHE'].replace(mapping)
df_lonsum['ANVENDELSE'] = df_lonsum['ANVENDELSE'].replace(mapping)

#set index
df_lonsum.set_index(['ANVENDELSE', 'TID'], inplace=True)
df_timerxselv.set_index(['BRANCHE', 'TID'], inplace=True)
df_timer_ialt.set_index(['BRANCHE', 'TID'], inplace=True)

#tjek
df_timer_ialt

INDHOLD
BRANCHE TID          
01000   2010    98594
10120   2010    79289
01000   1972   463150
10120   1972   187292
01000   1991   224577
...               ...
REST    2018  3934302
        2019  3972380
        2020  3827066
        2021  4062261
        2022  4225398

[171 rows x 1 columns]

In [9]:
df_timerxselv

INDHOLD
BRANCHE TID          
01000   2010    50879
10120   2010    77950
01000   1991    69831
10120   1991   124678
01000   1989    72936
...               ...
REST    2018  3672593
        2019  3728601
        2020  3594582
        2021  3801068
        2022  3960298

[171 rows x 1 columns]

In [10]:
# Sørg for at index-navnene er identiske, ellers kan Pandas brokke sig ved division
df_timerxselv.index.names = ['ANVENDELSE', 'TID']
df_timer_lon = df_timerxselv.reset_index()
df_timer_lon.columns = ['ANVENDELSE', 'TID', 'TIMER']
df_timer_lon.to_csv('Data69/Timer_lon_landbrugsdata.csv', index=False)
df_timer_ialt.index.names = ['ANVENDELSE', 'TID']
df_timer_output = df_timer_ialt.reset_index()
df_timer_output.columns = ['ANVENDELSE', 'TID', 'TIMER']
# 4. Gem til CSV
df_timer_output.to_csv('Data69/Timer_landbrugsdata.csv', index=False)
df_timer_output


,ANVENDELSE,TID,TIMER
0,01000,2010,98594
1,10120,2010,79289
2,01000,1972,463150
3,10120,1972,187292
4,01000,1991,224577
...,...,...,...
166,REST,2018,3934302
167,REST,2019,3972380
168,REST,2020,3827066
169,REST,2021,4062261


In [11]:
# 1. Udfør divisionen specifikt på 'INDHOLD' kolonnerne
df_timelon = df_lonsum['INDHOLD'] / (df_timerxselv['INDHOLD'] / 1000)

# 2. Gør det til et DataFrame og giv kolonnen et sigende navn
df_timelon = df_timelon.to_frame(name='TIMELOEN_KR')

# 3. FLYT INDEX TIL KOLONNER (Dette er løsningen på dit problem!)
# Når du gør dette, bliver 'ANVENDELSE' og 'TID' til rigtige kolonner i din CSV
df_timelon_output = df_timelon.reset_index()

# 4. Gem til CSV
df_timelon_output.to_csv('Data69/TimeLon_landbrugsdata.csv', index=False)

# Vis resultatet for at tjekke
df_timelon_output

,ANVENDELSE,TID,TIMELOEN_KR
0,01000,1966,4.961466
1,01000,1967,5.326186
2,01000,1968,5.644206
3,01000,1969,6.012333
4,01000,1970,5.843746
...,...,...,...
166,REST,2018,306.322765
167,REST,2019,311.951879
168,REST,2020,327.731151
169,REST,2021,327.713660


### Afgifter

In [12]:
# beregner toldsats ud fra løbende priser
df_loebende_afgift = df_input.loc[(df_input['TILGANG1'] == 'Produktskatter, netto ekskl. told og moms') & 
                         (df_input['PRISENHED'] == 'Løbende priser')]
df_loebende_moms = df_input.loc[(df_input['TILGANG1'] == 'Moms') & 
                         (df_input['PRISENHED'] == 'Løbende priser')]

# Drop kolonner
df_loebende_afgift = df_loebende_afgift.drop(columns=['PRISENHED', 'TILGANG1'])
df_loebende_moms = df_loebende_moms.drop(columns=['PRISENHED', 'TILGANG1'])


# beregner toldsats ud fra løbende priser
df_loebende_afgift = df_input.loc[(df_input['TILGANG1'] == 'Produktskatter, netto ekskl. told og moms') & 
                         (df_input['PRISENHED'] == 'Løbende priser')]
df_loebende_moms = df_input.loc[(df_input['TILGANG1'] == 'Moms') & 
                         (df_input['PRISENHED'] == 'Løbende priser')]

# Drop kolonner
df_loebende_afgift = df_loebende_afgift.drop(columns=['PRISENHED', 'TILGANG1'])
df_loebende_moms = df_loebende_moms.drop(columns=['PRISENHED', 'TILGANG1'])

# Find fælles kolonner (alle undtagen INDHOLD)
common_cols = [c for c in df_loebende_afgift.columns if c != 'INDHOLD']

# Merge på fælles kolonner for at sikre alle brancher kommer med
merged = df_loebende_afgift.merge(
    df_loebende_moms,
    on=common_cols,
    how='outer',  # outer join så alle brancher kommer med
    suffixes=('_afgift', '_moms')
)

# Læg sammen og håndter NaN
df_samlet_afgift = (merged['INDHOLD_afgift'].fillna(0) + merged['INDHOLD_moms'].fillna(0))

# Konverter til DataFrame
df_samlet_afgift_final = merged[common_cols].copy()
df_samlet_afgift_final['afgift'] = df_samlet_afgift

# Nu har du en DataFrame med alle brancher
df_samlet_afgift_final.to_csv('Data69/landbrugsdata_afgift.csv', index=False)
df_samlet_afgift_final


,ANVENDELSE,TID,afgift
0,01000 Landbrug og gartneri- (Anvendelse),1966,-95.392
1,01000 Landbrug og gartneri- (Anvendelse),1967,-64.073
2,01000 Landbrug og gartneri- (Anvendelse),1968,-90.042
3,01000 Landbrug og gartneri- (Anvendelse),1969,-123.646
4,01000 Landbrug og gartneri- (Anvendelse),1970,-20.129
...,...,...,...
166,REST_ANVENDELSE Øvrige brancher,2018,70253.174
167,REST_ANVENDELSE Øvrige brancher,2019,67811.271
168,REST_ANVENDELSE Øvrige brancher,2020,68645.137
169,REST_ANVENDELSE Øvrige brancher,2021,74630.676
